# Spark Exercise

Apache Spark is an excellent tool for data engineering projects due to its robust ability to process large-scale data efficiently through distributed computing. Spark's in-memory processing capabilities significantly enhance the speed of data operations, making it ideal for handling big data workloads. It supports various data sources and formats, offering versatility in data ingestion and transformation. Additionally, Spark's rich API supports multiple programming languages such as Python, Java, and Scala, catering to diverse developer preferences. Its ecosystem, which includes libraries for SQL, machine learning, and graph processing, provides a comprehensive suite for building complex data pipelines and analytics, making it a powerful and flexible choice for data engineering tasks.

Use Python, ```pyspark``` and ```pandas``` to explore Apache Spark RDD and DataFrame:

# Spark RDD

Spark RDD (Resilient Distributed Dataset) is a fundamental data structure in Apache Spark that enables fault-tolerant, distributed processing of large datasets across multiple nodes in a cluster. Spark RDDs provide a higher-level abstraction for performing distributed data processing tasks, including both map (transformations) and reduce (aggregations) operations.

## Import Necessary Libraries

In [ ]:
# Install required libraries
import subprocess
import sys

# Install PySpark
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
print("PySpark installed successfully!")

In [ ]:
# Path to DATA
PATH_DATA = "../data/raw/weather_raw_20260423_1641_start2013-01-01_end2025-12-31.json"
PATH_DATA_PROCESSED = "../data/processed/weather_processed_20260423_1641_start2013-01-01_end2025-12-31.json"

# Import necessary libraries
import json
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, max, min, year, month, dayofmonth
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import pandas as pd

# Verify paths exist
import os
print("Data file exists:", os.path.exists(PATH_DATA))

## Spark Context and Session
Initialize Spark Context and Spark Session

In [ ]:

#sc = SparkContext(master="local", appName="First App")
spark = SparkSession.builder \
    .appName("PySpark LinReg Scatter") \
    .master("spark://172.29.16.102:7077") \
    .getOrCreate()

## Load Data into RDD

In [ ]:
# Load raw JSON data into RDD
rdd_raw = spark.sparkContext.textFile(PATH_DATA)
# Filter out empty lines and load JSON
rdd_data = rdd_raw.filter(lambda x: x.strip() and x.startswith('{')) \
                   .map(lambda x: json.loads(x))
print(f"Total records in RDD: {rdd_data.count()}")
# Show first record to understand structure
print("Sample record:", rdd_data.take(1))

## Map Operation

Split data into individual parts and create key-value pairs

In [ ]:
# Map: Extract date and temperature, create key-value pairs (date -> temperature)
rdd_mapped = rdd_data.map(lambda record: (
    record.get('time', 'unknown'),  # Key: date/time
    (float(record.get('temperature_2m', 0)), 1)  # Value: (temperature, count)
))
print("Mapped RDD - first 5 records:")
for record in rdd_mapped.take(5):
    print(f"  {record}")

## Reduce Operation

Reduce your key-value pairs

In [ ]:
# Reduce: Sum temperatures and counts per date, then calculate average
rdd_reduced = rdd_mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average temperature per date
rdd_avg_temp = rdd_reduced.map(lambda x: (x[0], x[1][0] / x[1][1]))

print("Reduced RDD (average temperature per date) - first 5 records:")
for record in rdd_avg_temp.take(5):
    print(f"  Date: {record[0]}, Avg Temp: {record[1]:.2f}°C")

## Collect Results

Because of lazy evaluation, the map-reduce operation is performed only now. Show what you calculated.

In [ ]:
# Collect results to driver
results_rdd = rdd_avg_temp.collect()

print(f"Total dates processed: {len(results_rdd)}")
print("\nFirst 10 results (average temperature per date):")
for i, (date, avg_temp) in enumerate(results_rdd[:10], 1):
    print(f"  {i}. {date}: {avg_temp:.2f}°C")

## Save Results

In [ ]:
# Save RDD results to JSON
output_path = PATH_DATA_PROCESSED.replace('.parquet', '_rdd_results.parquet')

rdd_avg_temp.map(lambda x: json.dumps({"date": x[0], "avg_temperature_2m": x[1]})) \
            .saveAsTextFile(output_path)

print(f"RDD results saved to: {output_path}")

# Spark DataFrame

Spark DataFrame is a distributed collection of data organized into named columns, designed for efficient data manipulation and analysis in Apache Spark. It is used for various data processing tasks such as data ingestion, transformation, querying, and analysis in Apache Spark, providing a high-level abstraction that simplifies working with structured data.

## Load Data into DataFrame

In [ ]:
# Load JSON data into DataFrame
df = spark.read.json(PATH_DATA)
print("DataFrame loaded successfully!")
print(f"Number of rows: {df.count()}")

## View DataFrame Schema

In [ ]:
# Display DataFrame schema
print("DataFrame Schema:")
df.printSchema()

## View DataFrame Data

In [ ]:
# Display first 10 rows
print("First 10 rows of the DataFrame:")
df.show(10, truncate=False)

## Filter Data

Performe a filter operation on a column

In [ ]:
# Filter records where temperature is above 15°C
df_filtered = df.filter(col("temperature_2m") > 15.0)
print(f"Records with temperature > 15°C: {df_filtered.count()}")
print("\nFirst 5 filtered records:")
df_filtered.select("time", "temperature_2m", "relative_humidity_2m").show(5, truncate=False)

## Group By and Aggregate

Performe a group by and aggregat operation

In [ ]:
# Group by year and calculate aggregate statistics
df_with_year = df.withColumn("year", year(col("time")))

df_agg = df_with_year.groupBy("year").agg(
    avg("temperature_2m").alias("avg_temperature"),
    max("temperature_2m").alias("max_temperature"),
    min("temperature_2m").alias("min_temperature"),
    count("temperature_2m").alias("record_count")
).orderBy("year")

print("Average temperature statistics by year:")
df_agg.show(truncate=False)

## Save DataFrame to Parquet

In [ ]:
# Save DataFrame to Parquet format
agg_output_path = PATH_DATA_PROCESSED.replace('.parquet', '_dataframe_results.parquet')
df_agg.write.mode("overwrite").parquet(agg_output_path)
print(f"DataFrame aggregation results saved to: {agg_output_path}")

# Also save the filtered data
filtered_output_path = agg_output_path.replace('results', 'filtered')
df_filtered.write.mode("overwrite").parquet(filtered_output_path)
print(f"Filtered data saved to: {filtered_output_path}")